# 04 — Model Evaluation

Experimentation only — evaluates a fine-tuned sentiment model against TweetEval via `brandparadigm.sentiment.evaluate`. Neutral rows are dropped automatically (see docs/model_cards/roberta_sentiment.md). Requires a model already trained by notebook 03 / `scripts/run_training_sentiment.py`, and network access to reload its tokenizer.

In [ ]:
from brandparadigm.config.paths import CONFIGS_DIR
from brandparadigm.sentiment.dataset import build_tokenized_dataset
from brandparadigm.sentiment.evaluate import build_evaluation_report, load_tweeteval_eval_set
from brandparadigm.sentiment.model import load_trained_model_and_tokenizer
from brandparadigm.utils import read_yaml

sentiment_config = read_yaml(CONFIGS_DIR / "sentiment_config.yaml")
data_config = read_yaml(CONFIGS_DIR / "data_config.yaml")

model, tokenizer = load_trained_model_and_tokenizer(sentiment_config["training"]["output_dir"])
model.eval()

In [ ]:
eval_df = load_tweeteval_eval_set(data_config, split=sentiment_config["evaluation"]["split"])
print(f"{len(eval_df)} TweetEval examples after dropping Neutral rows")
eval_df["sentiment_label"].value_counts()

In [ ]:
import torch

eval_dataset = build_tokenized_dataset(
    eval_df, tokenizer, max_length=sentiment_config["model"]["max_seq_length"]
)
with torch.no_grad():
    logits = model(
        input_ids=eval_dataset["input_ids"], attention_mask=eval_dataset["attention_mask"]
    ).logits
y_pred = logits.argmax(dim=-1).numpy()
y_true = eval_dataset["labels"].numpy() if hasattr(eval_dataset["labels"], "numpy") else list(eval_dataset["labels"])

report = build_evaluation_report(y_true, y_pred)
report["classification_report"]